Approach 1: MSE‑Optimal Quantization (K‑Means–Style Bucketing)

This treats the problem like a 1‑D clustering problem:

Group FICO scores into K buckets

Minimize the squared error between each score and its bucket centroid

This is simple, stable, and works well when you only have FICO scores (no default labels).

In [1]:
import numpy as np
from sklearn.cluster import KMeans

def fico_buckets_mse(fico_scores, n_buckets):
    fico = np.array(fico_scores).reshape(-1, 1)

    kmeans = KMeans(n_clusters=n_buckets, random_state=42)
    kmeans.fit(fico)

    # Extract cluster centers and sort them
    centers = sorted(kmeans.cluster_centers_.flatten())

    # Boundaries are midpoints between centers
    boundaries = []
    for i in range(len(centers) - 1):
        boundaries.append((centers[i] + centers[i+1]) / 2)

    # Add min/max
    boundaries = [min(fico_scores)] + boundaries + [max(fico_scores)]

    # Create rating map (lower rating = better score)
    def map_score(score):
        for i in range(len(boundaries)-1):
            if boundaries[i] <= score < boundaries[i+1]:
                return i + 1
        return len(boundaries) - 1

    return boundaries, map_score


Approach 2: Log‑Likelihood–Optimal Bucketing (PD‑Aware)

This is more sophisticated and uses default information.

This produces buckets that:

Keep PD monotonic

Separate high‑risk and low‑risk borrowers

Are ideal for credit rating systems

Dynamic Programming Approach
We sort borrowers by FICO score, then use DP to find the optimal split points.

In [2]:
import numpy as np
import pandas as pd

def log_likelihood(n, k):
    p = k / n if n > 0 else 1e-9
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return k * np.log(p) + (n - k) * np.log(1 - p)

def fico_buckets_loglik(df, fico_col, default_col, K):
    df = df.sort_values(fico_col).reset_index(drop=True)
    N = len(df)

    # Precompute cumulative sums
    cum_n = np.arange(1, N+1)
    cum_k = df[default_col].cumsum()

    # DP tables
    dp = np.full((K+1, N+1), -np.inf)
    split = np.zeros((K+1, N+1), dtype=int)

    dp[0][0] = 0

    for k in range(1, K+1):
        for j in range(1, N+1):
            for i in range(k-1, j):
                n = j - i
                defaults = cum_k[j-1] - (cum_k[i-1] if i > 0 else 0)
                ll = log_likelihood(n, defaults)

                if dp[k-1][i] + ll > dp[k][j]:
                    dp[k][j] = dp[k-1][i] + ll
                    split[k][j] = i

    # Recover boundaries
    boundaries = []
    j = N
    for k in range(K, 0, -1):
        i = split[k][j]
        boundaries.append(df[fico_col].iloc[i])
        j = i

    boundaries = sorted(boundaries)
    boundaries = [df[fico_col].min()] + boundaries + [df[fico_col].max()]

    # Rating map
    def map_score(score):
        for i in range(len(boundaries)-1):
            if boundaries[i] <= score < boundaries[i+1]:
                return i + 1
        return len(boundaries) - 1

    return boundaries, map_score
